# Complexity-Weighted Ensemble Forecasting

This notebook demonstrates an online complexity-weighted ensemble combining simple models (naive last-value, moving average) and complex models across synthetic time series datasets (Random Walk and Sinusoidal Drift).
We track and compare forecasting performance metrics (MSE, MAE) across models.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
import numpy as np
import pandas as pd
import json
import os
import matplotlib.pyplot as plt

# NumPy 2.0 compatibility shims if needed
if not hasattr(np, 'alltrue'): np.alltrue = np.all
if not hasattr(np, 'sometrue'): np.sometrue = np.any
if not hasattr(np, 'product'): np.product = np.prod

np.random.seed(42)
print("Imports loaded successfully.")

In [ ]:
GITHUB_DATA_URL = 'https://raw.githubusercontent.com/AMGrobelnik/ai-invention-e14940-algorithmically-weighted-ensemble-foreca/main/round-2/experiment-1/demo/mini_demo_data.json'

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists('mini_demo_data.json'):
        with open('mini_demo_data.json') as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

data = load_data()
print(f"Loaded datasets: {[ds['dataset'] for ds in data['datasets']]}")

## Configuration
Set configuration parameters for evaluation.

In [ ]:
# Tunable parameters
CONFIG = {
    "max_examples_per_dataset": 25,
    "random_seed": 42
}
print("Configuration:", CONFIG)

## Evaluation and Error Analysis
Compute MSE and MAE across forecasting models (Naive, Moving Average, Complexity-Weighted Ensemble).

In [ ]:
results_summary = []

for ds in data['datasets']:
    ds_name = ds['dataset']
    examples = ds['examples'][:CONFIG["max_examples_per_dataset"]]
    
    y_true = []
    y_naive = []
    y_ma = []
    y_ensemble = []
    
    for ex in examples:
        y_true.append(float(ex["output"]))
        y_naive.append(float(ex["predict_naive"]))
        y_ma.append(float(ex["predict_moving_average"]))
        y_ensemble.append(float(ex["predict_complexity_weighted_ensemble"]))
    
    y_true = np.array(y_true)
    y_naive = np.array(y_naive)
    y_ma = np.array(y_ma)
    y_ensemble = np.array(y_ensemble)
    
    # Calculate MSE and MAE
    metrics = {
        "Dataset": ds_name,
        "Naive_MSE": np.mean((y_true - y_naive)**2),
        "Naive_MAE": np.mean(np.abs(y_true - y_naive)),
        "MA_MSE": np.mean((y_true - y_ma)**2),
        "MA_MAE": np.mean(np.abs(y_true - y_ma)),
        "Ensemble_MSE": np.mean((y_true - y_ensemble)**2),
        "Ensemble_MAE": np.mean(np.abs(y_true - y_ensemble))
    }
    results_summary.append(metrics)

df_results = pd.DataFrame(results_summary)
print(df_results.to_string(index=False))

## Visualization of Forecasting Performance
Plot MSE and MAE comparisons across models and datasets.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

datasets = df_results["Dataset"]
x = np.arange(len(datasets))
width = 0.25

# MSE Plot
axes[0].bar(x - width, df_results["Naive_MSE"], width, label='Naive')
axes[0].bar(x, df_results["MA_MSE"], width, label='Moving Average')
axes[0].bar(x + width, df_results["Ensemble_MSE"], width, label='Complexity Ensemble')
axes[0].set_ylabel('Mean Squared Error (MSE)')
axes[0].set_title('MSE Comparison across Datasets')
axes[0].set_xticks(x)
axes[0].set_xticklabels(datasets)
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.6)

# MAE Plot
axes[1].bar(x - width, df_results["Naive_MAE"], width, label='Naive')
axes[1].bar(x, df_results["MA_MAE"], width, label='Moving Average')
axes[1].bar(x + width, df_results["Ensemble_MAE"], width, label='Complexity Ensemble')
axes[1].set_ylabel('Mean Absolute Error (MAE)')
axes[1].set_title('MAE Comparison across Datasets')
axes[1].set_xticks(x)
axes[1].set_xticklabels(datasets)
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()